# Module 9 Part 1: AI agents

Two short demos.

1. **An agent learns by trial and error** (Q-learning).
2. **Two AI agents work together** (a writer and a reviewer).

## Part 1. An agent learns a route

The agent starts at **S** and must reach **G**. **T** is a trap.

```
 S  .  .  .
 .  T  .  .
 .  .  .  .
 .  .  .  G
```

Rewards:

- reach the goal: **+10** (game over)
- fall in the trap: **-10** (game over)
- any other move: **-1** (so short routes are better)

Nobody tells the agent where the goal is. It only sees the rewards.

In [1]:
import numpy as np

START = (0, 0)
TRAP = (1, 1)
GOAL = (3, 3)

ACTIONS = ["up", "down", "left", "right"]
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # how each action changes (row, column)

### The world: one move

Give it a cell and an action. It gives back the new cell, the reward, and whether the game is over.

In [2]:
def step(cell, action):
    # where would this move take the agent?
    row = cell[0] + MOVES[action][0]
    col = cell[1] + MOVES[action][1]

    # stay inside the 4 by 4 grid
    row = min(max(row, 0), 3)
    col = min(max(col, 0), 3)
    new_cell = (row, col)

    # reward, and is the game over?
    if new_cell == GOAL:
        return new_cell, 10, True
    if new_cell == TRAP:
        return new_cell, -10, True
    return new_cell, -1, False


# try one move: right from the start
print(step(START, 3))

((0, 1), -1, False)


### The Q-table: the agent's memory

One score for every cell and every action: 4 rows x 4 columns x 4 actions. All zero at the start.

A score means: *"if I take this action here, how much reward will I get from now on?"*

After every move the agent nudges one score:

$$
\text{new score} = \text{old score} + 0.5 \times (\text{target} - \text{old score})
$$

$$
\text{target} = \text{reward now} + 0.9 \times \text{best score from the new cell}
$$

- **0.5** is how fast it learns.
- **0.9** means future rewards count a bit less than rewards now.

**Exploring:** 20% of the time the agent picks a random action, so it keeps trying new things.

In [4]:
Q = np.zeros((4, 4, 4))          # Q[row, column, action]
rng = np.random.default_rng(42)  # fixed seed: same result every run

for episode in range(1000):
    cell = START

    for move in range(100):
        # 1. choose an action: random 20% of the time, otherwise the best score
        if rng.random() < 0.2:
            action = int(rng.integers(4))
        else:
            action = int(np.argmax(Q[cell]))

        # 2. make the move
        new_cell, reward, done = step(cell, action)

        # 3. update the score for (cell, action)
        if done:
            target = reward
        else:
            target = reward + 0.9 * np.max(Q[new_cell])
        old_score = Q[cell][action]
        Q[cell][action] = old_score + 0.5 * (target - old_score)

        # 4. move on
        cell = new_cell
        if done:
            break

#print("Trained for 1000 episodes")

### What did it learn?

The best action in every cell. Before running: which way should the arrows go?

In [5]:
ARROWS = ["^", "v", "<", ">"]

for row in range(4):
    line = ""
    for col in range(4):
        if (row, col) == GOAL:
            line = line + " G "
        elif (row, col) == TRAP:
            line = line + " T "
        else:
            best = int(np.argmax(Q[row, col]))
            line = line + " " + ARROWS[best] + " "
    print(line)

 >  >  v  v 
 v  T  >  v 
 >  v  v  v 
 >  >  >  G 


In [7]:
print(Q[0, 2].round(2))

[3.12 4.58 1.81 4.58]


In [8]:
# follow the arrows from the start, no random moves
cell = START

route = [cell]

for move in range(20):
    action = int(np.argmax(Q[cell]))
    cell, reward, done = step(cell, action)
    route.append(cell)
    if done:
        break

print("Route:", route)
print("Moves:", len(route) - 1)

Route: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 3)]
Moves: 6


**Takeaways**

- The agent found a 6-move route round the trap, from rewards alone.
- The Q-table is all it knows.

## Part 2. Two AI agents work together

An **agent** here = a language model + a role (the system prompt).

Same model, two roles:

- **Writer** writes a draft.
- **Reviewer** gives feedback.
- **Writer** improves the draft.

This is the **Reflection** pattern. Tools like CrewAI do the same thing with more structure.

**Setup (once):** `pip install anthropic python-dotenv`, then put `ANTHROPIC_API_KEY=your-key` in a file called `.env` next to this notebook. Never paste a key into a notebook.

In [9]:
import anthropic
from dotenv import load_dotenv

load_dotenv(".env", override=True)   # reads the key from .env, never printed
client = anthropic.Anthropic()        # finds the key by itself

In [10]:
def ask(role, message):
    # send one message to Claude, playing the given role
    response = client.messages.create(
        model="claude-opus-5-5",
        max_tokens=16000,
        output_config={"effort": "low"},   # quick and cheap
        system=role,
        messages=[{"role": "user", "content": message}],
    )

    # keep only the text of the reply
    text = ""
    for block in response.content:
        if block.type == "text":
            text = text + block.text
    return text

In [18]:
WRITER = "You are a friendly teacher. Explain ideas simply for beginners. Reply with the explanation only."
REVIEWER = "You review teaching material. Give at most three short bullet points: what is wrong, unclear or too hard."

TASK = "Explain Q-learning to a beginner in exactly three sentences."

In [13]:
# 1. the writer drafts
draft = ask(WRITER, TASK)
print("DRAFT:\n", draft, "\n")



DRAFT:
 Q-learning is a way for a computer program (called an agent) to learn good behavior through trial and error: it tries actions, sees what happens, and gets rewards or penalties, much like a pet learning tricks for treats. As it goes, the agent fills in a kind of "cheat sheet" called a Q-table, where each entry estimates how good it is to take a certain action in a certain situation, counting both the immediate reward and the rewards it can expect later. Over many tries, the agent updates these estimates and increasingly picks the actions with the highest scores, while still occasionally trying new things, until it learns a strategy that earns the most reward over time. 



In [14]:
# 2. the reviewer gives feedback (and knows the three-sentence limit)
feedback = ask(REVIEWER, "The writer was asked: " + TASK + "\n\nReview this:\n" + draft)
print("FEEDBACK:\n", feedback, "\n")



FEEDBACK:
 - **Missing the core idea:** "updates these estimates" is too vague. The defining step of Q-learning is nudging each score toward *the reward just received plus the best score available from the next situation*. Without this, the description fits almost any reinforcement-learning method.
- **Overclaims the outcome:** "until it learns a strategy that earns the most reward" implies a guaranteed result. Softer wording would be more accurate, e.g., "gradually learns a strategy that earns high reward."
- **Too dense for a beginner:** Each sentence packs in several ideas (exploration, delayed rewards, the Q-table). The "cheat sheet" metaphor also suggests the answers are given rather than learned, so "scorecard" might fit better. 



In [15]:
# 3. the writer improves the draft
final = ask(WRITER, TASK + "\n\nYour draft:\n" + draft + "\n\nFeedback:\n" + feedback + "\n\nWrite a better version.")
print("FINAL:\n", final)

FINAL:
 Q-learning is a way for a computer program, called an agent, to learn by trial and error, trying actions and earning rewards or penalties, much like a pet learning tricks for treats. The agent keeps a scorecard, called a Q-table, that rates how good each action is in each situation. After every action, it nudges that score toward the reward it just got plus the best score it expects from where it ended up, so over many tries it gradually learns a strategy that earns high reward.


In [16]:
import ollama

MODEL = "llama3.2"   # the model you downloaded with "ollama pull"

In [19]:
def ask(role, message):
    # the conversation: the role first, then our question
    conversation = [
        {"role": "system", "content": role},
        {"role": "user", "content": message},
    ]

    # send it to the model running on this computer
    response = ollama.chat(model=MODEL, messages=conversation)

    # keep only the text of the reply
    return response.message.content

In [20]:
# 1. the writer drafts
draft = ask(WRITER, TASK)
print("DRAFT:\n", draft, "\n")

# 2. the reviewer gives feedback (and knows the three-sentence limit)
feedback = ask(REVIEWER, "The writer was asked: " + TASK + "\n\nReview this:\n" + draft)
print("FEEDBACK:\n", feedback, "\n")

# 3. the writer improves the draft
final = ask(WRITER, TASK + "\n\nYour draft:\n" + draft + "\n\nFeedback:\n" + feedback + "\n\nWrite a better version.")
print("FINAL:\n", final)

DRAFT:
 Q-learning is an algorithm used for machine learning, which helps computers learn how to make decisions by exploring and interacting with their environment. It works by assigning a value, or "quality," to each state and action, so that the computer can decide which actions to take based on its estimated outcomes. As the computer tries different actions and experiences the results, it updates these values to improve its decision-making over time, allowing it to learn from trial and error. 

FEEDBACK:
 Here are three potential issues with this explanation:

* The definition of Q-learning is too general and doesn't specifically mention reinforcement learning, which is a crucial aspect of Q-learning.
* The phrase "estimated outcomes" could be clarified or rephrased for better understanding; it's not entirely clear what kind of outcomes are being estimated.
* The last sentence oversimplifies the process of updating values in Q-learning; it implies that the computer learns through di

**Takeaways**

- Each agent is the same model with a different system prompt.
- The "team" is just passing one agent's answer to the next.
- The reviewer is also an AI and can be wrong. A person still checks.
- Each call costs money: this demo makes 3.